# 10 — Seed runs (D030)
One person per model: set `EXP`, then Run all. The model is trained again with its existing settings and seeds 43 and 44 (seed 42 is the run from notebooks 05-07), so the final results are mean ± SD over 3 runs.
Finished runs are skipped and an interrupted one resumes, so after a disconnect just Run all again. Only the validation set is used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git -C /content/realwaste-classification pull -q || git clone -q https://github.com/KNV-DSEB/realwaste-classification.git /content/realwaste-classification
import sys
sys.path.insert(0, "/content/realwaste-classification")
for m in [m for m in sys.modules if m.split(".")[0] == "src"]:
    del sys.modules[m]  # re-import the code just pulled

In [ ]:
EXP = "E1"  # E1, E2 or E3

import torch
from src.dataset import load_frozen_split, use_local_copy
from src.tuning import plan, run_trial, tuning_table, write_selection
from src.utils import load_config

cfg = use_local_copy(load_config())
split_df = load_frozen_split(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp = device.type == "cuda"

In [ ]:
for phase in [1, 2, 3]:
    for overrides, seed in plan(cfg, EXP, phase):
        run_trial(cfg, EXP, overrides, seed, phase, split_df, device, amp)

print(tuning_table(cfg, EXP).to_string(index=False))
print(write_selection(cfg, EXP))